To understand how the histogram method calculates gain, you have to look at how it replaces the "brute force" search with a "binned" search.

In a standard (exact) tree, the algorithm looks at every single unique value of a feature to find the best split. In the **histogram method**, the algorithm simplifies the data first.

Here is the step-by-step process of how the gain is calculated using bins:

### 1. The Binning Step (Pre-processing)
Before the tree starts splitting, the algorithm takes a continuous feature (e.g., `Transaction_Amount`) and divides it into discrete bins.
*   **Raw Data:** `[1.2, 5.5, 10.1, 15.3, 22.0, 50.5, 100.2]`
*   **Bins:** `[0-10, 10-30, 30-110]`
*   **Binned Data:** `[Bin 0, Bin 0, Bin 1, Bin 1, Bin 1, Bin 2, Bin 2]`

Instead of tracking the exact value `1.2`, the algorithm only tracks that this data point belongs to `Bin 0`.

### 2. The Histogram Aggregation
For each bin, the algorithm calculates a **summary statistic**. It doesn't need the individual values; it only needs to know:
1.  **$S$ (Sum of weights/gradients):** The sum of the "errors" (gradients) of all points in that bin.
2.  **$H$ (Sum of squared gradients):** The sum of the squared "errors" (hessians) of all points in that bin.

**Example:**
If Bin 0 contains 100 samples, the algorithm calculates:
*   $\sum \text{Gradients in Bin 0} = 45.5$
*   $\sum \text{Hessians in Bin 0} = 100.0$

### 3. The Gain Calculation (The "Magic" Step)
The "Gain" is the measure of how much the error (loss) decreases if we split the data at a specific bin boundary. 

The formula for Gain used by XGBoost/LightGBM is:
$$\text{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L} + \frac{G_R^2}{H_R} - \frac{(G_L + G_R)^2}{H_L + H_R} \right] - \lambda$$

Where:
*   $G_L, H_L$: Sum of gradients and hessians in the **Left** child (all bins below the split).
*   $G_R, H_R$: Sum of gradients and hessians in the **Right** child (all bins above the split).
*   $\lambda$: Regularization parameter.

**How it works with bins:**
Instead of testing every possible value, the algorithm only tests the **boundaries between bins**.

1.  **Pick a bin boundary** (e.g., the line between Bin 1 and Bin 2).
2.  **Calculate Left Side:** Sum up all $G$ and $H$ for all bins $\le$ Bin 1.
3.  **Calculate Right Side:** Sum up all $G$ and $H$ for all bins $>$ Bin 1.
4.  **Plug into the Gain formula.**
5.  **Repeat** for the next bin boundary.
6.  **Pick the boundary** that yields the highest Gain.

### Summary: Why is this faster?
| Method | Complexity | What it does |
| :--- | :--- | :--- |
| **Exact (Brute Force)** | $O(\text{Data Points})$ | Checks every single value. If you have 10 million rows, it does 10 million calculations per feature. |
| **Histogram (`hist`)** | $O(\text{Number of Bins})$ | Checks only the bin boundaries. If you have 256 bins, it only does 256 calculations per feature, regardless of whether you have 1 million or 1 billion rows. |

**In short:** The algorithm converts the problem from "Where is the best split among these 1,000,000 points?" to "Which of these 256 bin boundaries provides the best Gain?"